# Exploring Chemical Space
Explore some clustering and visualisation models and compare them for two different datasets:
- A subset of ChEMBL small molecule entries with a molecular weight between 200 and 500. Since the entire set of ChEMBL entries is quite substantial, 10k entries were selected by random sampling (`chembl_200-500_10k`)
- The result of the PubChem search for "antibiotics" (`pubchem_antibiotics`).

Note: The ChEMBL dataset may still take quite some processing time - depending on your PC. You can do a random sampling of e.g. 3k entries in order to reduce the computational effort.

Tasks:
1) Load and inspect the two datasets `ChEMBL_200-500_10k.csv`. Note that they are fundamentally different.
2) Perform basic data cleaning and be mindful of which data to dismiss (if any). Hint: Think about standardising column names (at least for the relevant ones)
3) Make sure that the SMILES strings are valid - implement a function to clean up the SMILES returning (in a new column or Series) the canonical SMILES if the input is valid, and return `None` if the SMILES in the original data is not valid (). Hint: The `Normalizer` in  rdkit might be quite useful (https://www.rdkit.org/docs/cppapi/classRDKit_1_1MolStandardize_1_1Normalizer.html)
4) Calculate Morgan Fingerprints (radius 2, 2048 bit) from the SMILES strings via mol objects. Make sure not to use the outdated version. You can use either the dataframe, numpy arrays or simple lists for the fingerprints.
5) Run different clustering techniques, e.g. snippets provided for Butina and HDBSCAN. You can also try the scikit-learn models kmeans or dbscan.
6) Use the fingerprints to run UMAP and TSNE dimensionality reductions (snippets provided).
7) Plot the data in scatterplots, using the cluster labels as colour map.
8) Adjust some parameters of the clustering models and apply filters if needed (e.g. only visualise clusters of a size larger than 10) to reach some satisfactory result
9) Visualise a representative molecule (e.g. centroids or centers of clusters, or random :) ) of the three largest clusters for both methods using rdkit
10) Respond to the discussion points


Import dependencies and datasets

In [61]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem.MolStandardize.rdMolStandardize import Normalizer
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs
from rdkit.ML.Cluster import Butina
from rdkit.Chem import Draw

from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
import hdbscan
import umap
from sklearn.manifold import TSNE

In [34]:
df_chembl = pd.read_csv("chembl_200-500_10k.csv")
print(df_chembl.info())
df_chembl.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 29 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ChEMBL ID           10000 non-null  object 
 1   Name                182 non-null    object 
 2   Synonyms            82 non-null     object 
 3   Type                10000 non-null  object 
 4   Max Phase           75 non-null     float64
 5   Molecular Weight    10000 non-null  float64
 6   Targets             9593 non-null   float64
 7   Bioactivities       9593 non-null   float64
 8   AlogP               9946 non-null   float64
 9   Polar Surface Area  9946 non-null   float64
 10  HBA                 9946 non-null   float64
 11  HBD                 9946 non-null   float64
 12  #RO5 Violations     9946 non-null   float64
 13  #Rotatable Bonds    9946 non-null   float64
 14  Passes Ro3          9946 non-null   object 
 15  QED Weighted        9946 non-null   float64
 16  Aroma

,ChEMBL ID,Name,Synonyms,Type,Max Phase,Molecular Weight,Targets,Bioactivities,AlogP,Polar Surface Area,...,Heavy Atoms,Np Likeness Score,Molecular Formula,SMILES,Inchi Key,Inchi,Withdrawn Flag,Orphan,Records Key,Records Name
0,CHEMBL50524,NaN,NaN,Small molecule,NaN,361.47,2.0,2.0,1.96,84.66,...,25.0,-0.79,C18H23N3O3S,Nc1c(NC/C=C\COc2csc(CN3CCCCC3)c2)c(=O)c1=O,ZXLCEKMBIFPOSF-DJWKRKHSSA-N,InChI=1S/C18H23N3O3S/c19-15-16(18(23)17(15)22)...,False,-1,['12a'],['3-Amino-4-[(Z)-4-(5-piperidin-1-ylmethyl-thi...
1,CHEMBL1719619,NaN,NaN,Small molecule,NaN,339.35,4.0,4.0,2.33,110.62,...,25.0,-0.90,C18H17N3O4,COc1ccc(OC)c(C2C(C#N)=C(N)Oc3cc(C)nc(O)c32)c1,ATRZEYKVGNFNIL-UHFFFAOYSA-N,InChI=1S/C18H17N3O4/c1-9-6-14-16(18(22)21-9)15...,False,-1,"['SID859991', 'SID87345386']","['SID859991', 'SID87345386']"
2,CHEMBL1989505,NaN,NaN,Small molecule,NaN,417.54,2.0,2.0,6.02,92.93,...,30.0,-1.12,C23H23N5OS,Cc1ccc(NC(=O)Nc2ccc(-c3c(C(C)C)sc4ncnc(N)c34)c...,OCRAIYGHHHMVJB-UHFFFAOYSA-N,InChI=1S/C23H23N5OS/c1-13(2)20-18(19-21(24)25-...,False,-1,['SID103904394'],['SID103904394']
3,CHEMBL1673052,NaN,NaN,Small molecule,NaN,487.61,3.0,9.0,4.12,77.07,...,36.0,-1.28,C27H33N7O2,COc1cc(N2CCN(C)CC2)ccc1Nc1ncc2c(n1)N(C(C)C)c1c...,YYTFTZONXCQXLW-UHFFFAOYSA-N,InChI=1S/C27H33N7O2/c1-18(2)34-22-9-7-6-8-20(2...,False,-1,"['24', 'XMD10-78', 'Compound 18']",['11-isopropyl-2-((2-methoxy-4-(4-methylpipera...
4,CHEMBL3084770,NaN,NaN,Small molecule,NaN,426.56,1.0,1.0,3.58,70.84,...,31.0,-0.62,C24H34N4O3,CC[C@H]1COC[C@H](C)N1c1nc2c(C(=O)N[C@@H]3C[C@H...,UIKMHGOYAXJLIA-VPAKFMSCSA-N,InChI=1S/C24H34N4O3/c1-4-17-14-30-13-15(2)28(1...,False,-1,['35'],"['endo-2-((3S,5S)-3-ethyl-5-methylmorpholino)-..."


In [33]:
df_pubchem = pd.read_csv("pubchem_antibiotics.csv")
print(df_pubchem.info())
df_pubchem.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2358 entries, 0 to 2357
Data columns (total 38 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Compound_CID                        2358 non-null   int64  
 1   Name                                2358 non-null   object 
 2   Synonyms                            2330 non-null   object 
 3   Molecular_Formula                   2358 non-null   object 
 4   InChI                               2358 non-null   object 
 5   Smiles                              2358 non-null   object 
 6   InChIKey                            2358 non-null   object 
 7   IUPAC_Name                          2348 non-null   object 
 8   MeSH_Headings                       1124 non-null   object 
 9   Annotation_Content                  2225 non-null   object 
 10  Linked_BioAssays                    1093 non-null   object 
 11  Data_Source                         2358 no

,Compound_CID,Name,Synonyms,Molecular_Formula,InChI,Smiles,InChIKey,IUPAC_Name,MeSH_Headings,Annotation_Content,...,Defined_Atom_Stereo_Count,Undefined_Atom_Stereo_Count,Total_Bond_Stereo_Count,Defined_Bond_Stereo_Count,Undefined_Bond_Stereo_Count,Linked_PubChem_Literature_Count,Linked_PubChem_Patent_Count,Linked_PubChem_Patent_Family_Count,Annotation_Type_Count,Create_Date
0,16131155,Antibiotic A 47934,A-47934 Antibiotic|Antibiotic A 47934|RefChem:...,C58H44Cl3N7O21S,InChI=1S/C58H44Cl3N7O21S/c59-31-7-20-1-4-36(31...,C1[C@@H]2C(=O)N[C@@H](C3=CC(=CC(=C3)OC4=C(C=CC...,HRGFAEUWEMDRRZ-QLRHZSCISA-N,"(1S,2R,19R,22R,34S,37R,40R,52S)-22-amino-5,15,...",antibiotic A 47934,Classification|Literature|Patents|Taxonomy,...,8,0,0,0,0,17,60,13,4,20070703
1,6439108,Antibiotic S 632-B1,Antibiotic S 632-B1|121995-32-2|S632-B1|S632-B...,C17H25NO5,InChI=1S/C17H25NO5/c1-9(4-10(2)17-11(3)23-17)1...,CC1C(O1)/C(=C\C(C)C(=O)CC(CC2CC(=O)NC(=O)C2)O)/C,JEIOGENOOQCFDS-WMZJFQQLSA-N,4-[(Z)-2-hydroxy-5-methyl-7-(3-methyloxiran-2-...,antibiotic S 632-B1,Classification|Literature|Toxicity,...,0,4,1,1,0,3,0,0,3,20060428
2,9690107,Antibiotic FK 089,FK-089 antibiotic|Antibiotic FK 089|86070-74-8...,C14H12N4O7S2,InChI=1S/C14H12N4O7S2/c19-8(20)3-25-17-9(6-4-2...,C1C=C(N2[C@H](S1)[C@@H](C2=O)NC(=O)/C(=N\OCC(=...,YVVLVHFJBQEWHH-NSHRYQRRSA-N,"(6R,7R)-7-[[(2Z)-2-(carboxymethoxyimino)-2-(1,...",antibiotic FK 089,Biological Test Results|Classification|Literat...,...,2,0,1,1,0,2,5,1,4,20061024
3,125607,Antibiotic A447 C,Antibiotic A447 C|95599-38-5|Antibiotic A447-C...,C60H88N2O20,InChI=1S/C60H88N2O20/c1-12-60(70)26-41(79-46-2...,CCC1(CC(C2=C(C1OC3CC(C(C(O3)C)OC4CCC(C(O4)C)OC...,JCVKGZNZMVGUIP-UHFFFAOYSA-N,"7,10-bis[[4-(dimethylamino)-5-[5-(5-hydroxy-6-...",antibiotic A447 C,Classification|Literature,...,0,23,0,0,0,1,0,0,2,20050808
4,5487319,DOB-41 antibiotic,Dob-41 antibiotic|Antibiotic dob 41|115666-98-...,C19H18N2O6,InChI=1S/C19H18N2O6/c1-10(27-19(25)15(9-22)26-...,C[C@H](C1=C2C(=CC=C1)N=C3C(=N2)C=CC=C3C(=O)O)O...,OSEDIRANPWGFRX-MEBBXXQBSA-N,6-[(1R)-1-[(2R)-3-hydroxy-2-methoxypropanoyl]o...,DOB-41 antibiotic,Classification|Literature|Patents,...,2,0,0,0,0,1,5,1,3,20050808


In [ ]:
df_chembl_sel = df_chembl[[
    "Molecular Weight",
    "Polar Surface Area",
    "Heavy Atoms",
    "HBA",
    "HBD",
    "#Rotatable Bonds",
    "AlogP",
    "Molecular Formula",
    "SMILES"
]].copy()

df_chembl_sel.columns = [
    "molecular_weight",
    "polar_surface_area",
    "heavy_atoms",
    "h_bond_acceptors",
    "h_bond_donors",
    "rotatable_bonds",
    "logp",
    "molecular_formula",
    "smiles"
]

df_pubchem_sel = df_pubchem[[
    "Molecular_Weight",
    "Polar_Area",
    "Heavy_Atom_Count",
    "H-Bond_Acceptor_Count",
    "H-Bond_Donor_Count",
    "Rotatable_Bond_Count",
    "XLogP",
    "Molecular_Formula",
    "Smiles"
]].copy()

df_pubchem_sel.columns = [
    "molecular_weight",
    "polar_surface_area",
    "heavy_atoms",
    "h_bond_acceptors",
    "h_bond_donors",
    "rotatable_bonds",
    "logp",
    "molecular_formula",
    "smiles"
]

In [40]:
print(df_chembl_sel.info())
df_chembl_sel.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   molecular_weight    10000 non-null  float64
 1   polar_surface_area  9946 non-null   float64
 2   heavy_atoms         9946 non-null   float64
 3   h_bond_acceptors    9946 non-null   float64
 4   h_bond_donors       9946 non-null   float64
 5   rotatable_bonds     9946 non-null   float64
 6   logp                9946 non-null   float64
 7   molecular_formula   10000 non-null  object 
 8   smiles              9993 non-null   object 
dtypes: float64(7), object(2)
memory usage: 703.2+ KB
None


,molecular_weight,polar_surface_area,heavy_atoms,h_bond_acceptors,h_bond_donors,rotatable_bonds,logp,molecular_formula,smiles
0,361.47,84.66,25.0,7.0,2.0,8.0,1.96,C18H23N3O3S,Nc1c(NC/C=C\COc2csc(CN3CCCCC3)c2)c(=O)c1=O
1,339.35,110.62,25.0,7.0,2.0,3.0,2.33,C18H17N3O4,COc1ccc(OC)c(C2C(C#N)=C(N)Oc3cc(C)nc(O)c32)c1
2,417.54,92.93,30.0,5.0,3.0,4.0,6.02,C23H23N5OS,Cc1ccc(NC(=O)Nc2ccc(-c3c(C(C)C)sc4ncnc(N)c34)c...
3,487.61,77.07,36.0,8.0,1.0,5.0,4.12,C27H33N7O2,COc1cc(N2CCN(C)CC2)ccc1Nc1ncc2c(n1)N(C(C)C)c1c...
4,426.56,70.84,31.0,6.0,1.0,4.0,3.58,C24H34N4O3,CC[C@H]1COC[C@H](C)N1c1nc2c(C(=O)N[C@@H]3C[C@H...


In [41]:
print(df_pubchem_sel.info())
df_pubchem_sel.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2358 entries, 0 to 2357
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   molecular_weight    2358 non-null   float64
 1   polar_surface_area  2358 non-null   float64
 2   heavy_atoms         2358 non-null   int64  
 3   h_bond_acceptors    2358 non-null   int64  
 4   h_bond_donors       2358 non-null   int64  
 5   rotatable_bonds     2358 non-null   int64  
 6   logp                1905 non-null   float64
 7   molecular_formula   2358 non-null   object 
 8   smiles              2358 non-null   object 
dtypes: float64(3), int64(4), object(2)
memory usage: 165.9+ KB
None


,molecular_weight,polar_surface_area,heavy_atoms,h_bond_acceptors,h_bond_donors,rotatable_bonds,logp,molecular_formula,smiles
0,1313.4,459.0,90,22,15,3,0.7,C58H44Cl3N7O21S,C1[C@@H]2C(=O)N[C@@H](C3=CC(=CC(=C3)OC4=C(C=CC...
1,323.4,96.0,23,5,2,7,0.3,C17H25NO5,CC1C(O1)/C(=C\C(C)C(=O)CC(CC2CC(=O)NC(=O)C2)O)/C
2,412.4,212.0,27,11,3,7,0.0,C14H12N4O7S2,C1C=C(N2[C@H](S1)[C@@H](C2=O)NC(=O)/C(=N\OCC(=...
3,1157.3,273.0,82,22,6,15,6.1,C60H88N2O20,CCC1(CC(C2=C(C1OC3CC(C(C(O3)C)OC4CCC(C(O4)C)OC...
4,370.4,119.0,27,8,2,7,1.5,C19H18N2O6,C[C@H](C1=C2C(=CC=C1)N=C3C(=N2)C=CC=C3C(=O)O)O...


In [43]:
df_chembl_sel = df_chembl_sel.dropna()
df_chembl_sel = df_chembl_sel.drop_duplicates()
df_chembl_sel.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9946 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   molecular_weight    9946 non-null   float64
 1   polar_surface_area  9946 non-null   float64
 2   heavy_atoms         9946 non-null   float64
 3   h_bond_acceptors    9946 non-null   float64
 4   h_bond_donors       9946 non-null   float64
 5   rotatable_bonds     9946 non-null   float64
 6   logp                9946 non-null   float64
 7   molecular_formula   9946 non-null   object 
 8   smiles              9946 non-null   object 
dtypes: float64(7), object(2)
memory usage: 777.0+ KB


In [44]:
df_pubchem_sel = df_pubchem_sel.dropna()
df_pubchem_sel = df_pubchem_sel.drop_duplicates()
df_pubchem_sel.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1905 entries, 0 to 2357
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   molecular_weight    1905 non-null   float64
 1   polar_surface_area  1905 non-null   float64
 2   heavy_atoms         1905 non-null   int64  
 3   h_bond_acceptors    1905 non-null   int64  
 4   h_bond_donors       1905 non-null   int64  
 5   rotatable_bonds     1905 non-null   int64  
 6   logp                1905 non-null   float64
 7   molecular_formula   1905 non-null   object 
 8   smiles              1905 non-null   object 
dtypes: float64(3), int64(4), object(2)
memory usage: 148.8+ KB


Valdi SMILES strings

In [76]:
from rdkit.Chem.MolStandardize import rdMolStandardize

normalizer = rdMolStandardize.Normalizer()

def canonicalize_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = normalizer.normalize(mol)
    return Chem.MolToSmiles(mol, canonical=True)

df_chembl_sel["canonical_smiles"] = df_chembl_sel["smiles"].apply(canonicalize_smiles)
df_pubchem_sel["canonical_smiles"] = df_pubchem_sel["smiles"].apply(canonicalize_smiles)

[22:13:02] Initializing Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:13:02] Running Normalizer
[22:1

Calculate Morgan Fingerprints (`GetMorganGenerator`). You can experiment with other fingerprints (`GetRDKitFPGenerator`) as well and see how they impact the clusters.

In [88]:
from rdkit import Chem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator, GetRDKitFPGenerator

morgan_gen = GetMorganGenerator(radius=2, fpSize=2048)
rdkit_gen = GetRDKitFPGenerator(fpSize=2048)

df_chembl_sel["morgan_fp"] = df_chembl_sel["canonical_smiles"].apply(
    lambda s: morgan_gen.GetFingerprint(Chem.MolFromSmiles(s))
)

df_pubchem_sel["morgan_fp"] = df_pubchem_sel["canonical_smiles"].apply(
    lambda s: morgan_gen.GetFingerprint(Chem.MolFromSmiles(s))
)

#df_chembl_sel["rdkit_fp"] = df_chembl_sel["canonical_smiles"].apply(
#    lambda s: rdkit_gen.GetFingerprint(Chem.MolFromSmiles(s))
#)

#df_pubchem_sel["rdkit_fp"] = df_pubchem_sel["canonical_smiles"].apply(
#    lambda s: rdkit_gen.GetFingerprint(Chem.MolFromSmiles(s))
#)

In [ ]:
def smiles_to_morgan_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return morgan_gen.GetFingerprint(mol)

def fp_to_array(fp, nbits=2048):
    arr = np.zeros((nbits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def add_fingerprints(df):
    df = df.copy()
    df["morgan_fp"] = df["canonical_smiles"].apply(smiles_to_morgan_fp)
    df = df[df["morgan_fp"].notna()].reset_index(drop=True)
    X = np.array([fp_to_array(fp) for fp in df["morgan_fp"]], dtype=np.int8)
    return df, X

In [85]:
df_chembl_sel["morgan_fp"].info()


<class 'pandas.core.series.Series'>
Index: 9946 entries, 0 to 9999
Series name: morgan_fp
Non-Null Count  Dtype 
--------------  ----- 
9946 non-null   object
dtypes: object(1)
memory usage: 413.4+ KB


Butina Clustering: Investigate later how different cutoffs for the similarity affect the clusters.

In [80]:
df_chembl_sel_copy = df_chembl_sel.drop(columns=["smiles", "canonical_smiles"])

In [89]:
# Butina clustering requires distance matrix (distance = Tanimoto similarity); 
# fingerprints provided as list in this snippet - adjust as needed!
dists = []
fp_list = df_chembl_sel["morgan_fp"].tolist()
nfps = len(fp_list)

for i in range(1, nfps):
    similarities = DataStructs.BulkTanimotoSimilarity(fp_list[i], fp_list[:i])
    dists.extend([1-x for x in similarities])

# Apply Butina Clustering

# Apply different thresholds later and see how they affect the clustering
cutoff = 0.6  # Tanimoto similarity threshold; e.g. 04 for larger chemical families, 0.7 for tight analogues...

butina_clusters = Butina.ClusterData(
    dists, # similarity based distance matrix
    nfps, # number of fingerprints
    cutoff,
    isDistData=True
)

print("Number of clusters:", len(butina_clusters))

Number of clusters: 6790


In [90]:
# filter out small clusters, rare chemoptypes, ...
clusters_filtered = [c for c in butina_clusters if len(c) >= 10]

butina_labels = np.full(nfps, -1)
for cluster_id, cluster in enumerate(clusters_filtered):
    for id in cluster:
        butina_labels[id] = cluster_id

sizes = [len(c) for c in clusters_filtered]

print("clusters:", len(sizes))
print("mean size:", np.mean(sizes))
print("max size:", np.max(sizes))
print("singletons:", sum(s == 1 for s in sizes))

clusters: 30
mean size: 14.566666666666666
max size: 29
singletons: 0


HDBSCAN Clustering: Inspect how `min_cluster_size` and `min_samples` affect the clusters later on. This might depend quite a lot on the dataset (try 15 and 5 for the PubChem data, <10 and <5 for the ChEMBL)

In [70]:
# use HDBSCAN for clustering
hdbs_clustering = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=2,
    metric="jaccard"
)

hdbs_labels = hdbs_clustering.fit_predict(fp_list)
# print("Number of DBSCAN clusters:",
#       len(set(hdbs_labels)) - (1 if -1 in hdbs_labels else 0))
# print("Noise points:", list(hdbs_labels).count(-1))

In [71]:
print("Number of DBSCAN clusters:",
      len(set(hdbs_labels)) - (1 if -1 in hdbs_labels else 0))
print("Noise points:", list(hdbs_labels).count(-1))

Number of DBSCAN clusters: 214
Noise points: 8034


Embeddings: TSNE and UMAP

In [82]:
# convert fingerprints to numpy
X = np.zeros((nfps, 2048), dtype=int)
for i, fp in enumerate(fp_list):
    DataStructs.ConvertToNumpyArray(fp, X[i])

print(X)

[[0 0 0 ... 0 0 1]
 [1 0 1 ... 0 1 1]
 [1 1 0 ... 0 1 1]
 ...
 [1 0 1 ... 0 0 1]
 [0 0 0 ... 0 0 1]
 [1 1 1 ... 1 1 0]]


In [ ]:
# Dimensionality reduction by UMPA
umap_model = umap.UMAP(
    n_neighbors=25,
    min_dist=0.2,
    metric="jaccard",
    random_state=42
)

umap_embedding = umap_model.fit_transform(X)

/Users/raphaeland./opt/anaconda3/envs/DAC_env/lib/python3.8/site-packages/umap/umap_.py:1887: UserWarning: gradient function is not yet implemented for jaccard distance metric; inverse_transform will be unavailable
  warn(
/Users/raphaeland./opt/anaconda3/envs/DAC_env/lib/python3.8/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [59]:
# Dimensionality reduction by TSNE
tsne_model = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42
)

tsne_embedding = tsne_model.fit_transform(X)

Visualise the two clustering methods in the two embeddings (e.g. as subplot). Make sure to label the axes properly.

Visualise representative molecules of the three biggest clusters of both methods.

## Discussion points
1) What are the characteristics of the chemical spaces described in the two dataset? What is the difference?
2) How do density-based clustering techniques compare to models based on similarity in light of the differences in the datasets?
3) What were the best model parameters for the clustering techniques (i.e. that delivered a meaningful result)
4) Comment on the different dimensionality reduction techniques (again in light of the different dataset characteristics)
5) What was the best / most meaningful combination of dimensionality reduction and clustering methods?
6) Comment on some cheminformatics modelling challenges you may have encountered (e.g. runtime, singleton clusters, paramtersensitivity). What could be done to work around a large number of clusters of small size?
7) On the antibiotics dataset, can you identify some known antibiotic classes / motives in your clusters?
